# EnderLeaf script preparation

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Needed to import from the enderscope library

import os

os.chdir("..")

## Imports

In [ ]:
from typing import Literal

from rich.pretty import pprint

import numpy as np

from enderscope.scan_patterns import snake, plot_path
from enderscope.serial import Stage
from enderscope.bed import bed
import enderleaf.acquire as ea
import enderleaf.image as ei

In [ ]:
%matplotlib widget

## Constants

In [ ]:
TEMPLATE_LENGTH = 210
TEMPLATE_SIZE = (TEMPLATE_LENGTH, TEMPLATE_LENGTH)
ROW_COUNT, COL_COUNT = 9, 9
LEAF_DIAM = 17
CAM_RES = (4608, 2592)

## Scan Pattern

In [ ]:
positions = snake(cols=COL_COUNT, rows=ROW_COUNT) * [
    # steps
    TEMPLATE_LENGTH / COL_COUNT,
    TEMPLATE_LENGTH / ROW_COUNT,
] + [
    # origin
    TEMPLATE_LENGTH / COL_COUNT / 2, 
    TEMPLATE_LENGTH / ROW_COUNT / 2,
]
plot_path(
    positions,
    title="snake scan",
    field=(LEAF_DIAM + 3, LEAF_DIAM + 3),
    selected_rectangles=[17, 41, 55, 81],
    circle_diam=LEAF_DIAM,
)

## 3D Scan Demo

In [ ]:
s = Stage('virtual', 115200)

In [ ]:
s.home()

In [ ]:
def dummy_acquire_image(
    pos,
    read_qr: bool,
    resolution: tuple,
    focus_mode: Literal[ea.FocusMode.MANUAL, ea.FocusMode.HUNT, ea.FocusMode.AUTO],
    crop_data: ei.Rectangle | None = None,
):
    print("________________________________")
    s.move_position(pos)
    print(f"Moved to position {pos}")
    print(ea.capture(resulution=resolution, focus_mode=focus_mode, crop_data=crop_data))
    if read_qr is True:
        print("QR read")
    print("________________________________")

In [ ]:
dummy_acquire_image(
    pos=(bed.x_max / 2, bed.y_max / 2, bed.global_height),
    read_qr=True,
    resolution=CAM_RES,
    focus_mode=ea.FocusMode.AUTO,
)

for i, p in enumerate(positions):
    dummy_acquire_image(
        pos=np.append(p, bed.individual_height),
        read_qr=i == 0,
        resolution=CAM_RES,
        focus_mode=ea.FocusMode.HUNT,
        crop_data=ei.Rectangle(top=0, left=0, right=CAM_RES[0], bottom=CAM_RES[1])
        .shrink(new_height=1000, new_width=1000)
        .ensure_int(),
    )